## Route-Level Scoring

Builds `route_features.csv` — one row per Changi route (374 routes, 48 countries).
The trained RF model (calibrated on 8 countries with historical arrivals data) is applied
to every route to score fuel-price sensitivity. This delivers route-level suspension risk
rather than the original country-level aggregates.

In [1]:
import pandas as pd
import numpy as np
import pickle
import datetime
import math
from math import radians, sin, cos, sqrt, atan2

routes      = pd.read_csv('../data/processed/routes.csv')
airports    = pd.read_csv('../data/processed/airports.csv')
airline_types = pd.read_csv('../data/processed/airline_classifications_clean.csv')
gdp         = pd.read_csv('../data/processed/gdp_clean.csv')
fuel_monthly = pd.read_csv('../data/processed/fuel_monthly.csv')

print('Routes:', routes.shape)
print('Airports:', airports.shape)
print('Airlines classified:', airline_types.shape)
print('GDP rows:', gdp.shape)

Routes: (371, 4)
Airports: (85498, 19)
Airlines classified: (77, 2)
GDP rows: (5157, 4)


### Step 1 — distances via Haversine

In [2]:
SIN_LAT, SIN_LON = 1.3644, 103.9915

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

df = routes.merge(
    airports[['iata_code', 'iso_country', 'latitude_deg', 'longitude_deg']],
    left_on='destination_airport_iata', right_on='iata_code', how='left'
).drop(columns=['iata_code'], errors='ignore')

df['distance_km'] = df.apply(
    lambda r: haversine_km(SIN_LAT, SIN_LON, r['latitude_deg'], r['longitude_deg'])
    if pd.notna(r['latitude_deg']) else np.nan, axis=1
)

print('Missing distances:', df['distance_km'].isna().sum())
print(df[['airline_iata','destination_city','destination_country','distance_km']].head(5))

Missing distances: 0
  airline_iata destination_city destination_country  distance_km
0           SQ           Sydney           Australia  6294.727208
1           SQ        Melbourne           Australia  6033.904250
2           SQ         Brisbane           Australia  6143.832010
3           SQ            Perth           Australia  3912.358154
4           SQ         Adelaide           Australia  5403.776998


### Step 2 — carrier type (LCC / FSC)

In [3]:
df = df.merge(airline_types, on='airline_iata', how='left')
df['carrier_type'] = df['carrier_type'].fillna('Unknown')

print(df['carrier_type'].value_counts())

carrier_type
FSC        174
LCC        170
Unknown     30
Name: count, dtype: int64


### Step 3 — GDP per capita and YoY growth for each destination country

GDP data is annual, so YoY change is `pct_change(1)` (one year lag).

In [4]:
gdp['year'] = pd.to_numeric(gdp['year'], errors='coerce')
gdp = gdp.sort_values(['country', 'year'])
gdp['gdp_yoy_change'] = gdp.groupby('country')['gdp_per_capita'].pct_change(1) * 100

# Most recent observation per country
gdp_latest = gdp.groupby('country').last().reset_index()[['country', 'gdp_per_capita', 'gdp_yoy_change']]

# Map route destination country names -> World Bank country names
country_to_gdp = {
    'Australia':          'Australia',
    'Austria':            'Austria',
    'Bahrain':            'Bahrain',
    'Bangladesh':         'Bangladesh',
    'Belgium':            'Belgium',
    'Bhutan':             'Bhutan',
    'Brunei':             'Brunei Darussalam',
    'Cambodia':           'Cambodia',
    'Canada':             'Canada',
    'China':              'China',
    'Denmark':            'Denmark',
    'Ethiopia':           'Ethiopia',
    'Fiji':               'Fiji',
    'Finland':            'Finland',
    'France':             'France',
    'Germany':            'Germany',
    'Greece':             'Greece',
    'Hong Kong':          'Hong Kong SAR, China',
    'India':              'India',
    'Indonesia':          'Indonesia',
    'Italy':              'Italy',
    'Japan':              'Japan',
    'Macao':              'Macao SAR, China',
    'Malaysia':           'Malaysia',
    'Maldives':           'Maldives',
    'Mongolia':           'Mongolia',
    'Myanmar':            'Myanmar',
    'Nepal':              'Nepal',
    'Netherlands':        'Netherlands',
    'New Caledonia':      None,
    'New Zealand':        'New Zealand',
    'Oman':               'Oman',
    'Papua New Guinea':   'Papua New Guinea',
    'Philippines':        'Philippines',
    'Qatar':              'Qatar',
    'Saudi Arabia':       'Saudi Arabia',
    'South Africa':       'South Africa',
    'South Korea':        'Korea, Rep.',
    'Spain':              'Spain',
    'Sri Lanka':          'Sri Lanka',
    'Switzerland':        'Switzerland',
    'Taiwan':             None,
    'Thailand':           'Thailand',
    'Turkey':             'Turkiye',
    'United Arab Emirates': 'United Arab Emirates',
    'United Kingdom':     'United Kingdom',
    'United States':      'United States',
    'Vietnam':            'Viet Nam',
}

df['gdp_country'] = df['destination_country'].map(country_to_gdp)

df = df.merge(
    gdp_latest.rename(columns={'country': 'gdp_country'}),
    on='gdp_country', how='left'
)

# Taiwan: World Bank does not publish data for Taiwan.
# Using IMF/ADB estimate (~$35,000 USD), comparable to South Korea/Japan tier.
# Recent YoY growth ~2.5% based on Directorate-General of Budget, Accounting and Statistics (DGBAS).
df.loc[df['destination_country'] == 'Taiwan', 'gdp_per_capita'] = 35_000.0
df.loc[df['destination_country'] == 'Taiwan', 'gdp_yoy_change'] = 2.5

print(f'Routes with GDP data: {df["gdp_per_capita"].notna().sum()} / {len(df)}')
print(f'Missing GDP: {df[df["gdp_per_capita"].isna()][["destination_country"]].drop_duplicates()}')

Routes with GDP data: 372 / 374
Missing GDP:     destination_country
172       New Caledonia


### Step 4 — route-level LCC flag

Set `lcc_share = 1.0` if this specific route is operated by an LCC, `0.0` otherwise.
This is more precise than a country-level average: a Singapore Airlines route to Indonesia
should not inherit Indonesia's 71% LCC share — SQ is FSC and its passengers behave differently.

In [5]:
# Route-level binary: 1.0 = LCC, 0.0 = FSC or Unknown
df['lcc_share'] = (df['carrier_type'] == 'LCC').astype(float)

print('lcc_share distribution:')
print(df['lcc_share'].value_counts())
print()
# Confirm SQ (FSC) and TR (LCC) to same country now differ
indo = df[df['destination_country'] == 'Indonesia'][['airline_iata', 'carrier_type', 'lcc_share']].drop_duplicates()
print('Indonesia routes — lcc_share by carrier:')
print(indo.to_string(index=False))

lcc_share distribution:
lcc_share
0.0    204
1.0    170
Name: count, dtype: int64

Indonesia routes — lcc_share by carrier:
airline_iata carrier_type  lcc_share
          SQ          FSC        0.0
          TR          LCC        1.0
          GA          FSC        0.0
          QZ          LCC        1.0
          ID          LCC        1.0
          QG      Unknown        0.0
          IP      Unknown        0.0
          8B      Unknown        0.0


### Step 5 — calibrate fuel elasticity using the trained RF model

For each route, we evaluate the RF model at two fuel price points (baseline and baseline + $1/gal)
to compute the route's fuel-price elasticity: how many log-diff points does a $1/gal rise cost
this specific corridor? This is the calibration step — the model learned these relationships from
the 8 countries with historical arrivals data and we now apply them to all 374 routes.

In [6]:
with open('../models/rf_model.pkl', 'rb') as f:
    rf = pickle.load(f)

feature_cols = [
    'jet_fuel_usd_per_gallon', 'fuel_lag1', 'fuel_lag2', 'fuel_lag3',
    'fuel_change_3m', 'fuel_change_6m', 'fuel_volatility_6m',
    'gdp_per_capita', 'gdp_yoy_change', 'lcc_share',
    'month', 'quarter', 'is_peak_season'
]

# Use most recent actual fuel prices from historical data
fuel_monthly['year_month'] = pd.to_datetime(fuel_monthly['year_month'])
fuel_sorted = fuel_monthly.sort_values('year_month').reset_index(drop=True)
baseline_fuel = float(fuel_sorted['jet_fuel_usd_per_gallon'].iloc[-1])
fuel_6m_ago   = float(fuel_sorted['jet_fuel_usd_per_gallon'].iloc[-7])

now = datetime.datetime.now()
current_month   = now.month
current_quarter = math.ceil(now.month / 3)
is_peak         = int(now.month in [6, 7, 8, 12])

def make_features(routes_df, fuel_price):
    d = routes_df.copy()
    d['jet_fuel_usd_per_gallon'] = fuel_price
    d['fuel_lag1']               = fuel_price
    d['fuel_lag2']               = fuel_price
    d['fuel_lag3']               = fuel_price
    d['fuel_change_3m']          = (fuel_price - baseline_fuel) / baseline_fuel * 100
    d['fuel_change_6m']          = (fuel_price - fuel_6m_ago)   / fuel_6m_ago   * 100
    d['fuel_volatility_6m']      = 0.3
    d['month']                   = current_month
    d['quarter']                 = current_quarter
    d['is_peak_season']          = is_peak
    return d

df_base = make_features(df, baseline_fuel)
df_high = make_features(df, baseline_fuel + 1.0)

# Impute missing GDP rows with column medians so the model can still score them
X_base = df_base[feature_cols].fillna(df_base[feature_cols].median(numeric_only=True))
X_high = df_high[feature_cols].fillna(df_high[feature_cols].median(numeric_only=True))

df['rf_pred_at_baseline']      = rf.predict(X_base)
df['fuel_elasticity_per_dollar'] = rf.predict(X_high) - rf.predict(X_base)

print(f'Baseline fuel: ${baseline_fuel:.3f}/gal  |  6m ago: ${fuel_6m_ago:.3f}/gal')
print('\nFuel elasticity distribution (log-diff change per $1/gal rise):')
print(df['fuel_elasticity_per_dollar'].describe())

Baseline fuel: $4.119/gal  |  6m ago: $2.251/gal

Fuel elasticity distribution (log-diff change per $1/gal rise):
count    374.000000
mean       0.002504
std        0.003380
min       -0.003166
25%       -0.001040
50%        0.002682
75%        0.004784
max        0.009985
Name: fuel_elasticity_per_dollar, dtype: float64


### Step 6 — fuel cost impact per seat (economic context)

Computes the tangible dollar impact on per-seat fuel costs when the scenario price differs
from baseline. Assumes ~0.05 gallons fuel burn per seat-km (typical narrowbody / widebody average).

In [7]:
FUEL_BURN_GAL_PER_SEAT_KM = 0.05

df['fuel_cost_per_seat_baseline'] = (
    df['distance_km'] * FUEL_BURN_GAL_PER_SEAT_KM * baseline_fuel
).round(2)

# delta will be computed live in the dashboard from the slider value
print(df[['destination_city','destination_country','distance_km',
          'carrier_type','fuel_cost_per_seat_baseline',
          'fuel_elasticity_per_dollar']].sort_values('distance_km', ascending=False).head(10))

    destination_city destination_country   distance_km carrier_type  \
29          New York       United States  15339.009044          FSC   
30            Newark       United States  15335.209472          FSC   
27       Los Angeles       United States  14100.581341          FSC   
28     San Francisco       United States  13580.524449          FSC   
31           Seattle       United States  12977.936191          FSC   
294        Vancouver              Canada  12811.713806          FSC   
32         Vancouver              Canada  12811.713806          FSC   
13        Manchester      United Kingdom  10954.565285          FSC   
20         Barcelona               Spain  10898.109850          FSC   
278           London      United Kingdom  10881.725384          FSC   

     fuel_cost_per_seat_baseline  fuel_elasticity_per_dollar  
29                       3158.69                    0.005747  
30                       3157.90                    0.005747  
27                       2903

### Step 7 — sanity check: validate against the 8 training countries

In [8]:
# Old country-level predictions vs new route-level predictions for known countries
country_avg_old = pd.read_csv('../data/processed/country_avg_features.csv')
known_countries = country_avg_old['country'].tolist()

# Map old country names back to routes destination_country names
gdp_to_route = {v: k for k, v in country_to_gdp.items() if v is not None}
gdp_to_route.update({
    'France': 'France', 'Germany': 'Germany', 'Indonesia': 'Indonesia',
    'Japan': 'Japan', 'Malaysia': 'Malaysia', 'Philippines': 'Philippines',
    'Thailand': 'Thailand', 'United Kingdom': 'United Kingdom'
})

# New route-level avg prediction per country (avg across routes to that country)
route_country_avg = (
    df.groupby('destination_country')['rf_pred_at_baseline']
    .mean()
    .reset_index()
)

# Old country-level prediction
d_old = make_features(country_avg_old, baseline_fuel)
X_old = d_old[feature_cols].fillna(d_old[feature_cols].median(numeric_only=True))
country_avg_old['old_pred'] = rf.predict(X_old)

comparison = country_avg_old[['country','old_pred']].copy()
comparison['route_dest'] = comparison['country'].map(
    {c: c for c in ['France','Germany','Indonesia','Japan','Malaysia','Philippines','Thailand','United Kingdom']}
)
comparison = comparison.merge(route_country_avg, left_on='route_dest', right_on='destination_country', how='left')
comparison.columns = ['country','old_country_pred','route_dest','destination_country','new_route_avg_pred']
print('Country-level vs route-average predictions at baseline fuel:')
print(comparison[['country','old_country_pred','new_route_avg_pred']].round(4).to_string(index=False))

Country-level vs route-average predictions at baseline fuel:
       country  old_country_pred  new_route_avg_pred
        France            0.1462              0.1461
       Germany            0.1461              0.1464
     Indonesia            0.1059              0.0396
         Japan            0.1176              0.1159
      Malaysia            0.0621              0.0853
   Philippines           -0.1131              0.0458
      Thailand            0.0367              0.0554
United Kingdom            0.1481              0.1494


### Step 8 — save route_features.csv

In [9]:
keep_cols = [
    'airline_iata', 'destination_airport_iata', 'destination_city',
    'destination_country', 'iso_country', 'distance_km', 'carrier_type',
    'gdp_per_capita', 'gdp_yoy_change', 'lcc_share',
    'fuel_cost_per_seat_baseline', 'rf_pred_at_baseline', 'fuel_elasticity_per_dollar'
]

route_features = df[keep_cols].drop_duplicates(
    subset=['airline_iata', 'destination_airport_iata']
).copy()

route_features.to_csv('../data/processed/route_features.csv', index=False)

print(f'Saved route_features.csv: {len(route_features)} routes, {route_features["destination_country"].nunique()} countries')
print(route_features.head(5).to_string())

Saved route_features.csv: 371 routes, 48 countries
  airline_iata destination_airport_iata destination_city destination_country iso_country  distance_km carrier_type  gdp_per_capita  gdp_yoy_change  lcc_share  fuel_cost_per_seat_baseline  rf_pred_at_baseline  fuel_elasticity_per_dollar
0           SQ                      SYD           Sydney           Australia          AU  6294.727208          FSC    64603.985631       -0.698437        0.0                      1296.24             0.147898                    0.004784
1           SQ                      MEL        Melbourne           Australia          AU  6033.904250          FSC    64603.985631       -0.698437        0.0                      1242.53             0.147898                    0.004784
2           SQ                      BNE         Brisbane           Australia          AU  6143.832010          FSC    64603.985631       -0.698437        0.0                      1265.17             0.147898                    0.004784
3    